# Modern Coding Practice — LRU / TTL Cache (Amazon FAR style)

One domain, escalating constraints. Parts 1-3 build the core (concurrency at Part 3); Parts 4-5 are
stretch tasks (cache-stampede / single-flight under contention, then sharding + parallel warmup).
Fill stubs, run each test cell, peek at the collapsed `ref_` solutions only after trying.

`get` returns the value or `None` on miss. (Interview note: if `None` is a legal value you would use
a sentinel object instead — call that out.) **Clocks are injected** (`now` is passed in) for
deterministic TTL tests.

---

## Part 1 — LRU cache

`LRUCache(capacity)` with `get(key)`, `put(key, value)`, `size()`. Evict the least-recently-used
entry when capacity is exceeded. Both `get` and `put` count as a "use".

**Lock down:** O(1) get/put — hash map + recency ordering (an `OrderedDict`, or a dict + doubly
linked list). Does updating an existing key refresh recency? (Yes.)

In [9]:
from collections import OrderedDict
import time


class Node:
    def __init__(self, key, val, ttl):
        self.key = key
        self.val = val
        self.ttl = ttl
        self.prev = None
        self.next = None
class LinkedList:
    def __init__(self):
        self.head = Node(None, None, None)
        self.tail = self.head

    def add_node_to_end(self, node):
        self.tail.next = node
        node.prev = self.tail
        self.tail = node

    def insert(self, key, val, ttl) -> Node:
        node = Node(key, val, ttl)
        self.add_node_to_end(node)
        return node
        
    def update(self, node, ttl):
        prev = node.prev
        n = node.next
        node.ttl = ttl
        prev.next = n
        if n is not None:
            n.prev = prev
        self.add_node_to_end(node)
        return node

    def remove_head(self):
        node = self.head.next
        n = node.next
        key = node.key
        n.prev = self.head
        self.head.next = n
        del node
        return key

class LRUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self._d = {}
        self.ll = LinkedList()

    def get(self, key):
        if key in self._d:
            self.ll.update(self._d[key], time.monotonic())
            return self._d[key].val
        return None

    def put(self, key, value):
        if self.size() == self.cap:
            k = self.ll.remove_head()
            del self._d[k]
            
        if key not in self._d:
            self._d[key] = self.ll.insert(key, value, time.monotonic())
        else:
            node = self._d[key]
            node.val = value
            self.ll.update(self._d[key], time.monotonic())

    def size(self):
        return len(self._d)

In [10]:
# --- Part 1 tests ---
def _t1():
    c = LRUCache(2)
    c.put("a", 1)
    c.put("b", 2)
    assert c.get("a") == 1          # "a" is now most-recently-used
    c.put("c", 3)                   # capacity 2 -> evict LRU "b"
    assert c.get("b") is None
    assert c.get("a") == 1 and c.get("c") == 3
    c.put("a", 10)                  # update refreshes recency
    assert c.get("a") == 10 and c.size() == 2
    print("Part 1 OK")

_t1()

Part 1 OK


## Part 2 — Add TTL expiry

`TTLCache(capacity)` with `get(key, now)` and `put(key, value, now, ttl)`. An entry is valid while
`now < expiry` (where `expiry = now_at_put + ttl`) and expires when `now >= expiry`. Expired entries
are removed lazily on access. Capacity-based LRU eviction still applies.

**Lock down:** lazy (evict on access) vs active (background sweeper) expiry. `ttl <= 0` means the
entry is already expired. Why injecting `now` beats calling the clock internally (testability).

In [ ]:
class TTLCache:
    def __init__(self, capacity):
        raise NotImplementedError

    def get(self, key, now):
        raise NotImplementedError

    def put(self, key, value, now, ttl):
        raise NotImplementedError

    def size(self):
        raise NotImplementedError

class LRUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self._d = {}
        self.ll = LinkedList()

    def get(self, key, now):
        if key in self._d:
            if self._d[key].ttl < now:
                
            self.ll.update(self._d[key], now)
            return self._d[key].val
        return None

    def put(self, key, value, now, ttl):
        if self.size() == self.cap:
            k = self.ll.remove_head()
            del self._d[k]
            
        if key not in self._d:
            self._d[key] = self.ll.insert(key, value, time.monotonic())
        else:
            node = self._d[key]
            node.val = value
            self.ll.update(self._d[key], time.monotonic())

    def size(self):
        return len(self._d)

In [ ]:
# --- Part 2 tests ---
def _t2():
    c = TTLCache(capacity=10)
    c.put("a", 1, now=0, ttl=10)
    assert c.get("a", now=5) == 1
    assert c.get("a", now=10) is None          # expired at now >= 10
    c.put("b", 2, now=0, ttl=0)                # already expired
    assert c.get("b", now=0) is None
    c2 = TTLCache(capacity=2)
    c2.put("x", 1, now=0, ttl=100)
    c2.put("y", 2, now=0, ttl=100)
    assert c2.get("x", now=1) == 1             # refresh "x"
    c2.put("z", 3, now=1, ttl=100)             # evict LRU "y"
    assert c2.get("y", now=1) is None and c2.get("z", now=1) == 3
    print("Part 2 OK")

_t2()

## Part 3 — Thread-safe concurrent access

`ThreadSafeCache(cache)` wraps any cache so concurrent threads can `get`/`put`/`size` without
corrupting it. `OrderedDict`/linked-list mutation is not atomic, so guard it.

**The point:** 100 threads inserting 100 distinct keys into a capacity-50 cache must leave *exactly*
50 entries with no exception. Without the lock the recency structure corrupts mid-mutation.
**Discuss:** one lock vs read/write lock vs sharding (Part 5); this is coordination, not CPU work.

In [ ]:
import threading


class ThreadSafeCache:
    def __init__(self, cache):
        raise NotImplementedError

    def get(self, *args, **kwargs):
        raise NotImplementedError

    def put(self, *args, **kwargs):
        raise NotImplementedError

    def size(self):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    c = ThreadSafeCache(LRUCache(capacity=50))
    barrier = threading.Barrier(100)

    def worker(i):
        barrier.wait()
        c.put(i, i)

    ts = [threading.Thread(target=worker, args=(i,)) for i in range(100)]
    for t in ts:
        t.start()
    for t in ts:
        t.join()
    assert c.size() == 50          # capacity respected, no corruption
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Cache stampede / single-flight

When many threads miss the *same* key at once, a naive cache lets them all recompute it (a
"thundering herd"). Build `SingleFlightCache(capacity)`:

- `get_or_compute(key, now, ttl, compute) -> value`: on a hit, return it; on a miss, ensure
  `compute()` runs **at most once** across all concurrent callers for that key — the others wait and
  reuse the result.
- If `compute()` raises, wake the waiters and propagate (don't cache the failure); a later call may
  retry.
- `ttl <= 0` raises `ValueError`.

**Discuss:** per-key lock/Event vs one global lock, the recompute-on-failure policy, and the
hit/miss correctness window. This is the concurrency question that separates candidates.

In [ ]:
class SingleFlightCache:
    def __init__(self, capacity):
        raise NotImplementedError

    def get_or_compute(self, key, now, ttl, compute):
        raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    import time
    cache = SingleFlightCache(capacity=10)
    calls, clock_lock = [], threading.Lock()

    def compute():
        with clock_lock:
            calls.append(1)
        time.sleep(0.05)            # slow enough that all threads pile up on the miss
        return 42

    results, rlock, barrier = [], threading.Lock(), threading.Barrier(20)

    def worker():
        barrier.wait()
        v = cache.get_or_compute("k", now=0, ttl=100, compute=compute)
        with rlock:
            results.append(v)

    ts = [threading.Thread(target=worker) for _ in range(20)]
    for t in ts:
        t.start()
    for t in ts:
        t.join()
    assert results == [42] * 20
    assert len(calls) == 1, len(calls)          # stampede collapsed to one compute
    try:
        cache.get_or_compute("x", now=0, ttl=0, compute=lambda: 1)
    except ValueError:
        pass
    else:
        raise AssertionError("expected ValueError for ttl<=0")
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Sharded cache + parallel warmup

**(a)** `ShardedCache(num_shards, capacity_per_shard)` splits keys across shards (each its own LRU +
lock) so independent keys don't contend on one lock. `get(key)`, `put(key, value)`, `size()`.

**(b)** `parallel_warmup(keys) -> dict[key, value]` precomputes values for many keys (CPU-bound)
across processes with `ProcessPoolExecutor`; worker is `cachewk_workers.compute`. Then bulk-load.

**Discuss:** shard count vs core count, hot-key skew defeating sharding, GIL (warmup is CPU-bound ->
processes), and why warmup parallelizes cleanly (independent keys).

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import cachewk_workers


class ShardedCache:
    def __init__(self, num_shards, capacity_per_shard):
        raise NotImplementedError

    def get(self, key):
        raise NotImplementedError

    def put(self, key, value):
        raise NotImplementedError

    def size(self):
        raise NotImplementedError


def parallel_warmup(keys) -> dict:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    sc = ShardedCache(num_shards=4, capacity_per_shard=100)
    for i in range(50):
        sc.put(i, i * i)
    assert sc.get(10) == 100
    assert sc.get(999) is None
    assert sc.size() == 50
    warmed = parallel_warmup(list(range(8)))
    assert warmed == {i: i * i for i in range(8)}
    for k, v in warmed.items():
        sc.put(k, v)
    assert sc.get(7) == 49
    print("Part 5 OK")

_t5()

## Likely follow-ups
- LFU / ARC / 2Q eviction; W-TinyLFU. When LRU is the wrong policy.
- Active expiry (background sweeper) and memory pressure eviction.
- Distributed cache: consistent hashing, replication, invalidation, negative caching.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
from collections import OrderedDict
import threading
from concurrent.futures import ProcessPoolExecutor
import cachewk_workers


class RefLRU:
    def __init__(self, capacity):
        self.cap = capacity
        self.d = OrderedDict()

    def get(self, key):
        if key not in self.d:
            return None
        self.d.move_to_end(key)
        return self.d[key]

    def put(self, key, value):
        if key in self.d:
            self.d.move_to_end(key)
        self.d[key] = value
        if len(self.d) > self.cap:
            self.d.popitem(last=False)

    def size(self):
        return len(self.d)


class RefTTL:
    def __init__(self, capacity):
        self.cap = capacity
        self.d = OrderedDict()                 # key -> (value, expiry)

    def get(self, key, now):
        if key not in self.d:
            return None
        value, expiry = self.d[key]
        if now >= expiry:
            del self.d[key]
            return None
        self.d.move_to_end(key)
        return value

    def put(self, key, value, now, ttl):
        if key in self.d:
            self.d.move_to_end(key)
        self.d[key] = (value, now + ttl)
        if len(self.d) > self.cap:
            self.d.popitem(last=False)

    def size(self):
        return len(self.d)


class RefThreadSafeCache:
    def __init__(self, cache):
        self._c = cache
        self._lock = threading.Lock()

    def get(self, *a, **k):
        with self._lock:
            return self._c.get(*a, **k)

    def put(self, *a, **k):
        with self._lock:
            return self._c.put(*a, **k)

    def size(self):
        with self._lock:
            return self._c.size()


class RefSingleFlight:
    def __init__(self, capacity):
        self.cache = RefTTL(capacity)
        self.lock = threading.Lock()
        self.inflight = {}                     # key -> Event held by the leader

    def get_or_compute(self, key, now, ttl, compute):
        if ttl <= 0:
            raise ValueError("ttl must be positive")
        while True:
            with self.lock:
                val = self.cache.get(key, now)
                if val is not None:
                    return val
                ev = self.inflight.get(key)
                if ev is None:
                    ev = threading.Event()
                    self.inflight[key] = ev
                    leader = True
                else:
                    leader = False
            if not leader:
                ev.wait()                      # wait for the leader, then re-check the cache
                continue
            try:
                value = compute()
            except BaseException:
                with self.lock:
                    del self.inflight[key]
                    ev.set()                   # release waiters so one can retry
                raise
            with self.lock:
                self.cache.put(key, value, now, ttl)
                del self.inflight[key]
                ev.set()
            return value


class RefShardedCache:
    def __init__(self, num_shards, capacity_per_shard):
        self.n = num_shards
        self.shards = [RefLRU(capacity_per_shard) for _ in range(num_shards)]
        self.locks = [threading.Lock() for _ in range(num_shards)]

    def _idx(self, key):
        return hash(key) % self.n

    def get(self, key):
        i = self._idx(key)
        with self.locks[i]:
            return self.shards[i].get(key)

    def put(self, key, value):
        i = self._idx(key)
        with self.locks[i]:
            self.shards[i].put(key, value)

    def size(self):
        return sum(s.size() for s in self.shards)


def ref_parallel_warmup(keys):
    with ProcessPoolExecutor() as ex:
        return dict(ex.map(cachewk_workers.compute, keys))


_l = RefLRU(2)
_l.put("a", 1); _l.put("b", 2); _l.get("a"); _l.put("c", 3)
assert _l.get("b") is None and _l.get("a") == 1
_t = RefTTL(10); _t.put("a", 1, 0, 10)
assert _t.get("a", 5) == 1 and _t.get("a", 10) is None
assert ref_parallel_warmup([2, 3, 4]) == {2: 4, 3: 9, 4: 16}
print("reference solutions OK")